In [1]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier

# load data
df = pd.read_csv('../data/feature_engineered_data.csv')
print(df.shape)
df.head()

(99999, 28)


,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,q2_num_words,...,ctc_max,last_word_eq,first_word_eq,abs_len_diff,mean_len,longest_substr_ratio,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio
0,398782,496695,532029,what is the best marketing automation tool for...,what is the best marketing automation tool for...,1,75,76,13,13,...,0.923070,1.0,1.0,0.0,13.0,0.855263,99,99,99,99
1,115086,187729,187730,i am poor but i want to invest what should i do,i am quite poor and i want to be very rich wh...,0,48,56,13,16,...,0.466664,1.0,1.0,3.0,13.5,0.224490,69,67,65,74
2,327711,454161,454162,i am from india and live abroad i met a guy f...,t i e t to thapar university to thapar univers...,0,104,119,28,21,...,0.115384,0.0,0.0,6.0,23.0,0.047619,26,29,34,43
3,367788,498109,491396,why do so many people in the u s hate the sou...,my boyfriend doesnt feel guilty when he hurts ...,0,58,145,14,32,...,0.000000,0.0,0.0,17.0,21.5,0.050847,29,41,23,30
4,151235,237843,50930,consequences of bhopal gas tragedy,what was the reason behind the bhopal gas tragedy,0,34,49,5,9,...,0.333330,1.0,0.0,4.0,7.0,0.542857,55,70,48,69


In [2]:
# separate questions and features
questions_df = df[['question1', 'question2']].copy()
final_df     = df.drop(columns=['id', 'qid1', 'qid2', 'question1', 'question2'])
final_df     = final_df.reset_index(drop=True)
print(final_df.shape)

(99999, 23)


In [3]:
# tokenize questions
questions_df['q1_tokens'] = questions_df['question1'].apply(lambda x: str(x).lower().split())
questions_df['q2_tokens'] = questions_df['question2'].apply(lambda x: str(x).lower().split())
print("tokenization done")

tokenization done


In [4]:
# train word2vec
all_tokens = list(questions_df['q1_tokens']) + list(questions_df['q2_tokens'])

w2v = Word2Vec(
    sentences=all_tokens,
    vector_size=300,
    window=10,
    min_count=1,
    workers=4,
    epochs=30
)
print("word2vec done")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


word2vec done


In [14]:
# add this at the end of 03_ML_models.ipynb
w2v.save('../models/w2v_model.bin')
print("✅ Word2Vec saved!")

✅ Word2Vec saved!


In [5]:
# convert questions to vectors
def get_vector(tokens, model, size=300):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(size)

q1_vectors = np.array([get_vector(t, w2v) for t in questions_df['q1_tokens']], dtype=np.float32)
q2_vectors = np.array([get_vector(t, w2v) for t in questions_df['q2_tokens']], dtype=np.float32)

print("q1 vectors:", q1_vectors.shape)
print("q2 vectors:", q2_vectors.shape)

q1 vectors: (99999, 300)
q2 vectors: (99999, 300)


In [6]:
# create diff product and cosine features
diff    = q1_vectors - q2_vectors
product = q1_vectors * q2_vectors
cos_sim = np.array([
    cosine_similarity([q1_vectors[i]], [q2_vectors[i]])[0][0]
    for i in range(len(q1_vectors))
])

# convert to dataframe
DIM = 300
q1_df      = pd.DataFrame(q1_vectors, columns=[f'q1_w2v_{i}'      for i in range(DIM)])
q2_df      = pd.DataFrame(q2_vectors, columns=[f'q2_w2v_{i}'      for i in range(DIM)])
diff_df    = pd.DataFrame(diff,       columns=[f'diff_w2v_{i}'    for i in range(DIM)])
product_df = pd.DataFrame(product,    columns=[f'product_w2v_{i}' for i in range(DIM)])

print("features created")

features created


In [7]:
# combine all features
final_df['cosine_sim'] = cos_sim

full_df = pd.concat([
    final_df,
    q1_df, q2_df,
    diff_df, product_df
], axis=1).reset_index(drop=True)

print("final shape:", full_df.shape)

final shape: (99999, 1224)


In [8]:
# train test split
X = full_df.drop(columns=['is_duplicate'])
y = full_df['is_duplicate']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (79999, 1223)
Test: (20000, 1223)


In [11]:
import lightgbm as lgb
from lightgbm import LGBMClassifier

# train lightgbm
model = LGBMClassifier(
    n_estimators=1000,
    max_depth=-1,
    num_leaves=63,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(100)
    ]
)

# check accuracy
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred, target_names=['Not Duplicate', 'Duplicate']))

[LightGBM] [Info] Number of positive: 29428, number of negative: 50571
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.196133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 308919
[LightGBM] [Info] Number of data points in the train set: 79999, number of used features: 1223
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.367855 -> initscore=-0.541432
[LightGBM] [Info] Start training from score -0.541432
Training until validation scores don't improve for 30 rounds
[100]	valid_0's binary_logloss: 0.40517
[200]	valid_0's binary_logloss: 0.386279
[300]	valid_0's binary_logloss: 0.378892
[400]	valid_0's binary_logloss: 0.375023
[500]	valid_0's binary_logloss: 0.372279
[600]	valid_0's binary_logloss: 0.370456
[700]	valid_0's binary_logloss: 0.368516
[800]	valid_0's binary_logloss: 0.366821
[900]	valid_0's binary_logloss: 0.365867
Early stopping, best iteration is:
[961]	valid_0's binary_logloss: 0.3654

In [13]:
import pickle, os

os.makedirs('../models', exist_ok=True)

# save best model (lightgbm)
with open('../models/lgbm_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ LightGBM model saved!")

✅ LightGBM model saved!
